# optimizer-repr-string — worked example 2: Dynamic __repr__ from a defaults dict

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-repr-string`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When an optimizer stores its hyperparameters in a `defaults` dict (mirroring the PyTorch convention), `__repr__` can be built dynamically by iterating the dict keys. This makes the repr automatically include any new hyperparameters added later, without changing the `__repr__` implementation.

## Worked solution

**Step 1 — Store hyperparameters in `self.defaults`.**
In `__init__`, we build `self.defaults = {'lr': lr, 'weight_decay': weight_decay}`. This is the same approach `torch.optim.Optimizer` uses internally.

**Step 2 — Build the repr from the dict.**
In `__repr__`, we generate `key=value` pairs from `self.defaults.items()` and join them with `', '`. This is entirely data-driven: adding a new hyperparameter to `defaults` automatically includes it in the repr.

**Step 3 — Verify the output.**
We check the exact string produced. We also add a new hyperparameter to `defaults` after construction and confirm it appears in the next `repr()` call.

**Step 4 — Verify exclusion of non-scalar state.**
Buffers and param lists are stored separately, not in `defaults`, so they never appear in the repr.

In [ ]:
import torch as t

class DynamicSGD:
    def __init__(self, params, lr=0.01, weight_decay=0.0):
        self.params   = list(params)
        self.defaults = {'lr': lr, 'weight_decay': weight_decay}

    def __repr__(self):
        hparams = ', '.join(f'{k}={v}' for k, v in self.defaults.items())
        return f'DynamicSGD({hparams})'

# --- exercise it ---
t.manual_seed(0)
params = [t.tensor([1.0, 2.0], requires_grad=True)]
opt = DynamicSGD(params, lr=0.05, weight_decay=1e-4)
rep = repr(opt)
print(rep)  # DynamicSGD(lr=0.05, weight_decay=0.0001)
assert rep == 'DynamicSGD(lr=0.05, weight_decay=0.0001)'

# Adding a new hparam automatically appears in repr
opt.defaults['momentum'] = 0.9
new_rep = repr(opt)
print(new_rep)  # DynamicSGD(lr=0.05, weight_decay=0.0001, momentum=0.9)
assert 'momentum=0.9' in new_rep
print('Dynamic repr works!')